# Task-Regressed BOLD Functional Connectivity
Estimate parcel-level background BOLD connectivity for one session. First prepare and review the fMRIPrep derivatives following the [execution guide](../../docs/ds006033.md), then install the repository with the fMRI extra in the notebook kernel. The pipeline regresses modeled task responses and nuisance signals and censors high-motion frames.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "pyproject.toml").is_file())
dataset = root / "data" / "ds006033"  # Edit for a different download location
subject = "sub-02"
derivatives = root / "data" / "fmriprep"  # Directory containing the subject folders
output = root / "results" / "generated" / "ds006033" / "notebook-fmri"
session = "ses-01"

In [ ]:
prefix = f"{subject}_{session}_task-innerspeech"
func = dataset / subject / session / "func"
events = pd.read_csv(func / f"{prefix}_events.tsv", sep="\t")
display(events.head())
display(events["trial_type"].value_counts().rename("event count"))

In [ ]:
subprocess.run([
    sys.executable, "-m", "inner_speech.ds006033.fmri",
    str(dataset), str(derivatives), str(output),
    "--subject", subject, "--session", session,
], check=True)

In [ ]:
session_output = output / subject / session
display(json.loads((session_output / "qc.json").read_text()))
frames = pd.read_csv(session_output / "frame_qc.csv")
display(frames["retained"].value_counts().rename("frame count"))

matrix = pd.read_csv(session_output / "background_connectivity.csv", index_col=0)
fig, ax = plt.subplots()
image = ax.imshow(matrix.to_numpy(), cmap="RdBu_r", vmin=-1, vmax=1)
ax.set(title="Task-regressed BOLD connectivity", xlabel="Schaefer parcel index",
       ylabel="Schaefer parcel index")
fig.colorbar(image, ax=ax, label="Pearson r")
plt.show()